# Prairie Vigie – notebook 5 : indice de vulnérabilité communal

**Choix fixés :**
1. L'indice ne contient que des **facteurs de risque**. Natura 2000 et zones humides sont des **enjeux** : ils restent des filtres de
   priorité (notebook 4).
2. Quatre indicateurs, connus fin 2024 (RPG jusqu'à 2023), orientés « plus grand = plus vulnérable » :
   - `faible_part_pp` = 1 − part des prairies permanentes dans la surface déclarée de la commune en 2023
     (sens choisi d'après le modèle à la parcelle : plus de prairies autour = moins de conversions) ;
   - `recul_pp_2019_2023` = baisse relative de la surface en prairies permanentes entre 2019 et 2023
     (2019 et non 2018 : définition stable des prairies permanentes à partir de 2019) ;
   - `taux_conv_2019_2023` = part des prairies converties lors des campagnes 2019→20 à 2022→23 (l'« historique » du test 5 bis) ;
   - `proximite_cereales` = − distance (km) à la commune « céréalière » la plus proche, définie comme ≥ 70 % de terres arables
     dans au moins 100 ha déclarés en 2023 (communes normandes seulement : les zones céréalières des régions voisines ne sont pas prises en compte).
3. **Indice** = moyenne des rangs (0 à 1) des quatre indicateurs, poids égaux, **sans aucun réglage**.
4. **Test** identique au test 5 bis : communes avec au moins 50 ha de prairies en 2023 ; cible = part des prairies 2023 converties
   lors de la campagne 2024→25 ; mesure = corrélation de Spearman ; intervalle à 95 % par bootstrap (1 000 tirages, graine 0).
5. **Règle de décision** : l'indice est adopté s'il dépasse l'historique seul d'au moins **0,05** de corrélation. Sinon : « testé, non retenu ».

In [ ]:
# 0. Même dossier que les notebooks 1 à 4
DOSSIER = "datathon_prairies"
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")                                   # point de montage standard de Colab
    os.makedirs(os.path.join("/content/drive/MyDrive", DOSSIER), exist_ok=True)
    os.chdir(os.path.join("/content/drive/MyDrive", DOSSIER))
except ImportError:
    pass                                                            # hors Colab : dossier courant
print("Dossier de travail :", os.getcwd())
if not os.path.exists("sorties_prairies"):
    print("⚠ Dossier vide : lancer d'abord les notebooks 1 à 4 dans ce même dossier.")

In [ ]:
# 0 bis. Bibliothèques
!pip -q install geopandas pyogrio pyarrow

In [ ]:
# 1. Paramètres (fixés avant les résultats)
import time
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
T_DEBUT = time.time()

DEPS = ["14", "27", "50", "61", "76"]
CACHE, SORTIES = Path("cache_rpg"), Path("sorties_prairies")
MIN_HA_COMMUNE = 50          # communes retenues pour le test (même seuil que le test 5 bis)
SEUIL_CEREALES = 0.70        # part de terres arables qui définit une commune « céréalière »
MIN_HA_CEREALES = 100        # surface déclarée minimale pour être classée céréalière
REGLE = 0.05                 # gain de corrélation exigé face à l'historique
N_BOOT, GRAINE = 1000, 0

PP, HERBE, GEL = 18, {17, 18, 19}, {11, 12, 13}
ARABLE = {1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 15, 16, 24, 25, 26}
def classe(g):
    g = int(g) if pd.notna(g) else -1
    return ("pp" if g == PP else "herbe" if g in HERBE else "arable" if g in ARABLE
            else "gel" if g in GEL else "autre")

COMMUNES = gpd.GeoDataFrame(pd.concat([gpd.read_parquet(CACHE / f"communes_{d}.parquet").assign(dep=d) for d in DEPS],
                                      ignore_index=True), geometry="geometry", crs=2154)
print(len(COMMUNES), "communes")

In [ ]:
# 2. Cible et historique
TAB = pd.concat([pd.read_parquet(SORTIES / f"variables_{d}.parquet").assign(dep=d) for d in DEPS], ignore_index=True)
NOUV = pd.concat([pd.read_parquet(SORTIES / f"v4_{d}.parquet").assign(dep=d) for d in DEPS], ignore_index=True)
TOUT = pd.concat([TAB[TAB.a0 < 2022], NOUV], ignore_index=True)
passe = TOUT[TOUT.a0 <= 2022]                                   # campagnes 2019→20 à 2022→23, connues fin 2024
test = TOUT[(TOUT.a0 == 2023) & TOUT.y2.notna()]                 # prairies 2023, conversion lors de la campagne 2024→25

h = passe.assign(c=passe.y * passe.surface_ha).groupby("commune").agg(ha=("surface_ha", "sum"), c=("c", "sum"))
T = test.assign(o=test.y2 * test.surface_ha).groupby("commune").agg(n_pp=("surface_ha", "size"),
                                                                    ha=("surface_ha", "sum"), o=("o", "sum"))
T = T[T.ha >= MIN_HA_COMMUNE].join((h.c / h.ha).rename("taux_conv_2019_2023"), how="inner")
T["observe"] = T.o / T.ha
rho_hist = T.taux_conv_2019_2023.corr(T.observe, method="spearman")
print(f"{len(T)} communes ; historique seul : Spearman = {rho_hist:.2f} (notebook 4, test 5 bis : 0,29)")

In [ ]:
# 3. Indicateurs communaux tirés du RPG en cache (2019 et 2023), surface rattachée à la commune du point intérieur de la parcelle
def surfaces_commune(annee):
    out = []
    for dep in DEPS:
        g = gpd.read_parquet(CACHE / f"rpg_{dep}_{annee}_v3.parquet", columns=["code_group", "dans_dep", "geometry"])
        g = g[g["dans_dep"].astype(bool)]
        pts = gpd.GeoDataFrame({"classe": g["code_group"].apply(classe).values, "ha": (g.area / 1e4).values},
                               geometry=g.representative_point().values, crs=2154)
        j = gpd.sjoin(pts, COMMUNES.loc[COMMUNES.dep == dep, ["code", "geometry"]], how="left", predicate="within")
        out.append(j[~j.index.duplicated()])
    s = pd.concat(out).pivot_table(index="code", columns="classe", values="ha", aggfunc="sum", fill_value=0)
    s["total"] = s.sum(axis=1)
    return s

S23, S19 = surfaces_commune(2023), surfaces_commune(2019)
I = pd.DataFrame(index=S23.index)
I["part_pp_2023"] = S23["pp"] / S23["total"]
I["faible_part_pp"] = 1 - I["part_pp_2023"]
pp19 = S19["pp"].reindex(I.index)
I["recul_pp_2019_2023"] = ((pp19 - S23["pp"]) / pp19).where(pp19 > 0)
I["part_arable_2023"] = S23.get("arable", 0) / S23["total"]

# Distance du point intérieur de la commune à la commune céréalière la plus proche (0 si la commune l'est elle-même)
cer = COMMUNES[COMMUNES.code.isin(I.index[(I.part_arable_2023 >= SEUIL_CEREALES) & (S23["total"] >= MIN_HA_CEREALES)])]
pts = gpd.GeoDataFrame({"code": COMMUNES.code.values}, geometry=COMMUNES.representative_point().values, crs=2154)
d = gpd.sjoin_nearest(pts, cer[["geometry"]], how="left", distance_col="dist_m")
d = d[~d.index.duplicated()].set_index("code")["dist_m"]
I["distance_cereales_km"] = (d.reindex(I.index) / 1000).round(2)
I["proximite_cereales"] = -I["distance_cereales_km"]
print(f"{len(cer)} communes céréalières (≥ {SEUIL_CEREALES:.0%} de terres arables, ≥ {MIN_HA_CEREALES} ha)")
print(I[["part_pp_2023", "recul_pp_2019_2023", "distance_cereales_km"]].describe().round(3).to_string())

In [ ]:
# 4. Test : indice contre historique (règle fixée avant le résultat)
X = T.join(I, how="inner")
IND = ["faible_part_pp", "recul_pp_2019_2023", "taux_conv_2019_2023", "proximite_cereales"]
X = X.dropna(subset=IND)                       # quelques communes sans prairie en 2019 : recul non calculable
rg = X[IND].rank(pct=True)
X["indice_expert"] = rg.mean(axis=1)
X["indice_sans_historique"] = rg.drop(columns="taux_conv_2019_2023").mean(axis=1)

def rho(col, df):
    return df[col].corr(df["observe"], method="spearman")

rng = np.random.default_rng(GRAINE)
ecarts = []
for _ in range(N_BOOT):
    b = X.sample(len(X), replace=True, random_state=int(rng.integers(2**31 - 1)))
    ecarts.append(rho("indice_expert", b) - rho("taux_conv_2019_2023", b))
lo, hi = np.percentile(ecarts, [2.5, 97.5])

RES = pd.DataFrame([{"methode": m, "communes": len(X), "spearman": round(rho(m, X), 3)} for m in
                    ["taux_conv_2019_2023", "indice_expert", "indice_sans_historique",
                     "faible_part_pp", "recul_pp_2019_2023", "proximite_cereales"]])
gain = rho("indice_expert", X) - rho("taux_conv_2019_2023", X)
ADOPTE = bool(gain >= REGLE)
print(RES.to_string(index=False))
print(f"\nGain de l'indice sur l'historique : {gain:+.3f} (IC 95 % bootstrap {lo:+.3f} à {hi:+.3f}) ; règle : ≥ +{REGLE}")
print("Décision :", "indice ADOPTÉ" if ADOPTE else "indice testé, NON RETENU : la carte garde l'historique observé")
# Le gain, son intervalle et la décision ne concernent que la ligne de l'indice
est_indice = RES.methode == "indice_expert"
RES.loc[est_indice, "gain_vs_historique"] = round(gain, 3)
RES.loc[est_indice, "ic95_bas"], RES.loc[est_indice, "ic95_haut"] = round(lo, 3), round(hi, 3)
RES.loc[est_indice, "adopte"] = ADOPTE
RES.to_csv(SORTIES / "indice_vulnerabilite_test.csv", index=False)

# Indicateurs par commune : publiables si l'indice est adopté (faits mesurés, masqués sous 5 prairies), internes sinon
C = X[["n_pp", "ha", "part_pp_2023", "recul_pp_2019_2023", "taux_conv_2019_2023", "distance_cereales_km", "indice_expert"]] \
      .rename(columns={"ha": "ha_pp_2023"}).round(4).reset_index().rename(columns={"index": "commune"})
C.loc[C.n_pp < 5, C.columns[2:]] = np.nan
C.to_csv(SORTIES / ("indice_vulnerabilite_communes.csv" if ADOPTE else "indice_vulnerabilite_communes_INTERNE.csv"), index=False)

In [ ]:
# 5. Ajout au dossier publiable et nouveau contrôle
import re, shutil
PUB = Path("publiable")
if PUB.exists():
    a_copier = ["indice_vulnerabilite_test.csv"] + (["indice_vulnerabilite_communes.csv"] if ADOPTE else [])
    for f in a_copier:
        shutil.copy(SORTIES / f, PUB / "donnees" / f)
    if not ADOPTE:
        (PUB / "donnees" / "indice_vulnerabilite_communes.csv").unlink(missing_ok=True)
    # Dictionnaire des données : ajout de la description du nouveau fichier
    dico = PUB / "donnees" / "DICTIONNAIRE_DES_DONNEES.md"
    LIGNE = ("- Notebook 5 : indice_vulnerabilite_test.csv (test de l'indice de vulnérabilité communal : "
             "corrélation de Spearman avec les conversions 2024→25, gain face à l'historique, intervalle à 95 %, décision)")
    if dico.exists() and "indice_vulnerabilite_test" not in dico.read_text(encoding="utf-8"):
        with open(dico, "a", encoding="utf-8") as fd:
            fd.write("\n" + LIGNE + "\n")
    INTERDIT = re.compile("|".join(["pac" + "age", r"\bi" + r"up\b", "num_" + "parcel", "num_" + "ilot", "/content/", "MyDrive"]), re.I)
    problemes = [f.name for f in (PUB / "donnees").iterdir() if INTERDIT.search(f.read_text(encoding="utf-8", errors="ignore"))]
    problemes += [f.name for f in (PUB / "donnees").glob("*.csv")
                  if set(pd.read_csv(f, nrows=0).columns) & {"pid", "score", "top10", "p", "y", "y2", "cx1", "cy1"}]
    print("Ajoutés :", a_copier, "| contrôle :", problemes or "OK")
    shutil.make_archive("publiable", "zip", PUB)
else:
    print("⚠ Dossier publiable absent : lancer d'abord le notebook 4.")

## B. Second indice, tiré des études régionales : la « pression fourragère » (test exploratoire)
**Pourquoi.** Deux études de la statistique agricole décrivent qui retourne les prairies :
- Agreste Normandie, *Études* n° 6 (DRAAF Normandie, septembre 2021), prairies 2008-2019 : les exploitations **laitières**
  (spécialisées ou en polyculture-élevage) portent l'essentiel des pertes, concentrées dans le bocage laitier de l'ouest ;
- DRAAF Bretagne (2025) : les prairies retournées par les élevages laitiers et mixtes deviennent surtout du **maïs**
  (15 800 ha sur 21 700 ha en 2022-2023), en lien avec l'agrandissement des troupeaux.

Le nombre de vaches laitières par commune n'est pas en données ouvertes ; le maïs déclaré dans le RPG sert d'indicateur indirect
de cette pression fourragère.

1. `part_mais_2023` = part du maïs (grain et ensilage, groupe 2 du RPG) dans la surface déclarée de la commune en 2023 ;
2. `hausse_mais_2019_2023` = évolution de cette part entre 2019 et 2023 ;
3. `taux_conv_2019_2023` = l'historique, comme dans la partie A.

**Indice de pression fourragère** = moyenne des rangs des trois indicateurs, poids égaux, sans réglage. Même test et même règle que la
partie A : adopté s'il dépasse l'historique seul d'au moins 0,05.

In [ ]:
# 6. Second indice : pression fourragère (maïs)
def part_mais(annee):
    out = []
    for dep in DEPS:
        g = gpd.read_parquet(CACHE / f"rpg_{dep}_{annee}_v3.parquet", columns=["code_group", "dans_dep", "geometry"])
        g = g[g["dans_dep"].astype(bool)]
        pts = gpd.GeoDataFrame({"mais": (g["code_group"] == 2).values, "ha": (g.area / 1e4).values},
                               geometry=g.representative_point().values, crs=2154)
        j = gpd.sjoin(pts, COMMUNES.loc[COMMUNES.dep == dep, ["code", "geometry"]], how="left", predicate="within")
        out.append(j[~j.index.duplicated()])
    j = pd.concat(out)
    tot = j.groupby("code")["ha"].sum()
    return j[j.mais].groupby("code")["ha"].sum().reindex(tot.index).fillna(0) / tot

M23, M19 = part_mais(2023), part_mais(2019)
X["part_mais_2023"] = M23.reindex(X.index)
X["hausse_mais_2019_2023"] = (M23 - M19.reindex(M23.index)).reindex(X.index)
IND_B = ["taux_conv_2019_2023", "part_mais_2023", "hausse_mais_2019_2023"]
XB = X.dropna(subset=IND_B).copy()
rgB = XB[IND_B].rank(pct=True)
XB["indice_pression_fourragere"] = rgB.mean(axis=1)
XB["pression_sans_historique"] = rgB.drop(columns="taux_conv_2019_2023").mean(axis=1)

rng = np.random.default_rng(GRAINE)
ecarts = []
for _ in range(N_BOOT):
    b = XB.sample(len(XB), replace=True, random_state=int(rng.integers(2**31 - 1)))
    ecarts.append(rho("indice_pression_fourragere", b) - rho("taux_conv_2019_2023", b))
loB, hiB = np.percentile(ecarts, [2.5, 97.5])
RESB = pd.DataFrame([{"methode": m, "communes": len(XB), "spearman": round(rho(m, XB), 3)} for m in
                     ["taux_conv_2019_2023", "indice_pression_fourragere", "pression_sans_historique",
                      "part_mais_2023", "hausse_mais_2019_2023"]])
gainB = rho("indice_pression_fourragere", XB) - rho("taux_conv_2019_2023", XB)
ADOPTE_B = bool(gainB >= REGLE)
print(RESB.to_string(index=False))
print(f"\nGain de l'indice de pression fourragère sur l'historique : {gainB:+.3f} (IC 95 % {loB:+.3f} à {hiB:+.3f}) ; règle : ≥ +{REGLE}")
print("Décision :", "indice ADOPTÉ (test exploratoire, à confirmer sur la campagne suivante)" if ADOPTE_B
      else "indice testé, NON RETENU : la carte garde l'historique observé")
est = RESB.methode == "indice_pression_fourragere"
RESB.loc[est, "gain_vs_historique"] = round(gainB, 3)
RESB.loc[est, "ic95_bas"], RESB.loc[est, "ic95_haut"] = round(loB, 3), round(hiB, 3)
RESB.loc[est, "adopte"] = ADOPTE_B
RESB.to_csv(SORTIES / "indice_pression_fourragere_test.csv", index=False)

if PUB.exists():
    shutil.copy(SORTIES / "indice_pression_fourragere_test.csv", PUB / "donnees" / "indice_pression_fourragere_test.csv")
    LIGNE_B = ("- Notebook 5 : indice_pression_fourragere_test.csv (test exploratoire d'un indice fondé sur le maïs déclaré, "
               "d'après les études Agreste Normandie 2021 et DRAAF Bretagne 2025 : corrélations, gain face à l'historique, décision)")
    if dico.exists() and "indice_pression_fourragere_test" not in dico.read_text(encoding="utf-8"):
        with open(dico, "a", encoding="utf-8") as fd:
            fd.write(LIGNE_B + "\n")
    f = PUB / "donnees" / "indice_pression_fourragere_test.csv"
    ok = not INTERDIT.search(f.read_text(encoding="utf-8"))
    print("Ajouté au dossier publiable | contrôle :", "OK" if ok else "PROBLÈME")
    shutil.make_archive("publiable", "zip", PUB)
print(f"Durée du notebook : {(time.time() - T_DEBUT) / 60:.1f} min")